# Hafta 10 — Kümeleme (Clustering)

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

Kümelemede elimizde doğru cevaplar yoktur. Örneğin bir mağaza, müşterilerini alışveriş alışkanlıklarına göre gruplara ayırmak ister ama bu grupların neler olduğunu önceden bilmez. Kümeleme algoritmaları benzer müşterileri kendiliğinden bir araya toplar. Sonuçta "indirim dönemlerinde alışveriş yapanlar" veya "her ay düzenli gelenler" gibi gruplar ortaya çıkar.

En bilinen yöntem olan k-ortalamalar (k-means) şöyle çalışır: Önce rastgele birkaç merkez noktası seçilir ve her müşteri kendisine en yakın merkezin grubuna katılır. Ardından her merkez, kendi grubunun tam ortasına taşınır. Gruplar artık değişmeyene kadar bu iki adım tekrarlanır.

---

## Bu Haftanın Hedefleri

- k-means algoritmasını elle çalıştırmak ve sıfırdan kodlamak
- Hiyerarşik kümelemeyi ve dendrogramı anlamak
- DBSCAN ile yoğunluk tabanlı kümeleme yapmak
- Küme sayısını dirsek yöntemi ve siluet skoruyla seçmek

---

## 1. k-means

### Çalışma mantığı

```
1. k tane merkez seç
2. Tekrarla:
   a. ATAMA:      Her noktayı en yakın merkezin kümesine ata
   b. GÜNCELLEME: Her merkezi, kümesindeki noktaların ortalamasına taşı
3. Atamalar değişmeyince dur
```

Algoritma, küme içi kareler toplamını (**WCSS**) küçültür.

### Elle örnek (k = 2)

Noktalar: P1 (1,1), P2 (2,1), P3 (1,2), P4 (5,4), P5 (6,5), P6 (5,6). Başlangıç merkezleri: c₁ = P1, c₂ = P2.

| Adım | Küme 1 | Küme 2 | Yeni merkezler |
|---|---|---|---|
| 1. iterasyon | P1, P3 | P2, P4, P5, P6 | c₁ = (1, 1.5), c₂ = (4.5, 4) |
| 2. iterasyon | P1, **P2**, P3 | P4, P5, P6 | c₁ = (1.33, 1.33), c₂ = (5.33, 5) |
| 3. iterasyon | değişiklik yok | değişiklik yok | **yakınsadı** |

İkinci iterasyonda P2, yeni c₁ merkezine (mesafe 1.12) c₂'den (mesafe 3.91) daha yakın olduğu için küme değiştirdi.

### Sıfırdan k-means

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def kmeans(X, k, iterasyon=100, seed=0):
    rng = np.random.default_rng(seed)
    merkez = X[rng.choice(len(X), k, replace=False)]
    for _ in range(iterasyon):
        mesafe = ((X[:, None, :] - merkez[None, :, :]) ** 2).sum(axis=2)   # (n, k)
        etiket = mesafe.argmin(axis=1)                                      # ATAMA
        yeni = np.array([X[etiket == j].mean(axis=0) for j in range(k)])   # GÜNCELLEME
        if np.allclose(yeni, merkez):
            break
        merkez = yeni
    return etiket, merkez

from sklearn.datasets import make_blobs
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=3)
etiket, merkez = kmeans(X, k=4)
plt.scatter(X[:, 0], X[:, 1], c=etiket, s=10, cmap="tab10")
plt.scatter(merkez[:, 0], merkez[:, 1], c="k", marker="X", s=200); plt.show()

### Dikkat edilmesi gerekenler

- **Başlangıca bağlıdır:** Kötü seçilen merkezler kötü bir sonuca yol açabilir. Çözüm olarak algoritma birkaç kez çalıştırılır ve merkezler birbirinden uzak seçilir (**k-means++**). scikit-learn bunu otomatik yapar.
- Kümelerin **yuvarlak ve benzer büyüklükte** olduğunu varsayar; ay veya halka biçimli kümeleri bulamaz.
- k önceden belirtilmelidir ve **ölçekleme şarttır**.

---

## 2. Hiyerarşik Kümeleme

Her nokta tek başına bir küme olarak başlar. Her adımda **birbirine en yakın iki küme birleştirilir**. Sonunda tek bir küme kalır. Bu birleşmelerin ağacına **dendrogram** denir.

İki küme arasındaki mesafe farklı biçimlerde ölçülebilir: en yakın noktalar arası (tekli bağlantı), en uzak noktalar arası (tam bağlantı) veya kümeyi en az dağıtan birleşme (Ward).

**Elle örnek** (tek boyut, tam bağlantı): A = 1, B = 2, C = 4, D = 7, E = 8

| Adım | Birleşen kümeler | Mesafe |
|---|---|---|
| 1 | A + B | 1 |
| 2 | D + E | 1 |
| 3 | {A, B} + C | 3 |
| 4 | {A, B, C} + {D, E} | 7 |

```
mesafe
  7 │      ┌──────────┐
  3 │   ┌──┴──┐       │       ← burada kesersek 2 küme: {A,B,C} ve {D,E}
  1 │  ┌┴┐    │     ┌─┴─┐
    │  A B    C     D   E
```

Dendrogram yatay bir çizgiyle kesilerek küme sayısı seçilir. Uzun dikey çizgiler (büyük mesafe sıçramaları) doğal ayrımları gösterir.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

noktalar = np.array([[1], [2], [4], [7], [8]])
dendrogram(linkage(noktalar, method="complete"), labels=list("ABCDE")); plt.show()

---

## 3. DBSCAN: Yoğunluk Tabanlı Kümeleme

DBSCAN'e göre küme, noktaların **yoğun** olduğu bölgedir. İki parametre vardır: komşuluk yarıçapı **ε** ve bir noktanın "çekirdek" sayılması için gereken en az komşu sayısı **minPts**.

- **Çekirdek nokta:** ε yarıçapında en az minPts nokta bulunur.
- **Sınır nokta:** Kendisi çekirdek değildir ama bir çekirdeğin komşusudur.
- **Gürültü:** Hiçbir çekirdeğin komşusu değildir.

**Elle örnek:** Noktalar 1, 2, 3, 8, 9, 10, 20; ε = 1.5, minPts = 3

- 2'nin komşuları {1, 2, 3} → çekirdek. 1 ve 3 sınır noktadır. → **Küme 1: {1, 2, 3}**
- 9'un komşuları {8, 9, 10} → çekirdek. → **Küme 2: {8, 9, 10}**
- 20'nin komşusu yok → **gürültü**

In [ ]:
from sklearn.cluster import DBSCAN
print(DBSCAN(eps=1.5, min_samples=3).fit_predict([[1], [2], [3], [8], [9], [10], [20]]))
# [ 0  0  0  1  1  1 -1]    (-1 = gürültü)

DBSCAN küme sayısını kendisi bulur, her şekildeki kümeyi yakalar ve aykırı noktaları ayırır.

---

## 4. Küme Sayısını Seçmek

**Dirsek yöntemi:** k = 1, 2, 3, ... için WCSS çizilir. Eğrinin "dirsek" yaptığı, yani yeni bir kümenin artık az fayda sağladığı nokta seçilir.

**Siluet skoru:** Her nokta için:

```
a = kendi kümesindeki noktalara ortalama mesafe      (ne kadar uyumlu?)
b = en yakın diğer kümedeki noktalara ortalama mesafe (ne kadar ayrık?)
s = (b − a) / max(a, b)          1'e yakın: iyi kümelenmiş · 0: sınırda · negatif: yanlış kümede
```

**Elle örnek:** Bölüm 1'deki P1 = (1, 1) için a = (1 + 1) / 2 = 1.00 ve b = (5.00 + 6.40 + 6.40) / 3 = 5.93 olur. Buradan s = (5.93 − 1.00) / 5.93 = **0.83** çıkar; P1 çok iyi kümelenmiştir.

---

## 5. Laboratuvar: Müşteri Segmentasyonu

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

rng = np.random.default_rng(42)
def grup(n, gelir, harcama):
    return pd.DataFrame({"YillikGelir": rng.normal(*gelir, n), "HarcamaSkoru": rng.normal(*harcama, n)})
df = pd.concat([grup(40, (25, 6), (20, 8)), grup(40, (25, 6), (80, 8)), grup(80, (55, 8), (50, 7)),
                grup(40, (88, 12), (82, 8)), grup(40, (88, 12), (17, 8))], ignore_index=True)
X = StandardScaler().fit_transform(df)

ks = range(2, 10)
wcss = [KMeans(k, n_init=10, random_state=0).fit(X).inertia_ for k in ks]
sil = [silhouette_score(X, KMeans(k, n_init=10, random_state=0).fit_predict(X)) for k in ks]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ks, wcss, "o-"); ax[0].set_title("Dirsek yöntemi")
ax[1].plot(ks, sil, "s-"); ax[1].set_title("Siluet skoru")
plt.show()

df["Segment"] = KMeans(5, n_init=10, random_state=0).fit_predict(X)
print(df.groupby("Segment").mean().round(1))

**Görevler:**
1. Dirsek ve siluet grafiklerine göre hangi k'yı seçersiniz?
2. Her segmente bir isim verin (ör. "yüksek gelir, düşük harcama → temkinli müşteriler") ve bir pazarlama önerisi yazın.
3. Aynı veride DBSCAN deneyin. Kaç küme ve kaç gürültü noktası buldu?

---

## 6. Özet

| | k-means | Hiyerarşik | DBSCAN |
|---|---|---|---|
| k gerekli mi? | Evet | Hayır (dendrogramdan seçilir) | Hayır |
| Küme şekli | Yuvarlak | Bağlantı türüne bağlı | Her şekil |
| Gürültü | Her noktayı bir kümeye atar | Her noktayı atar | Gürültüyü ayırır |

## Kendinizi Sınayın

1. k-means'te neden WCSS'yi en küçük yapan k değerini seçemeyiz?
2. DBSCAN örneğinde ε = 6 seçilseydi sonuç ne olurdu?
3. Bölüm 1'deki P4 = (5, 4) için siluet değerini hesaplayın.

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.